# Adult Income Prediction Pipeline


### Mount Google Drive

This cell mounts your Google Drive, allowing the notebook to access files stored there, such as your dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Install and Import Libraries

This cell installs necessary Python libraries for data manipulation, machine learning, imbalance handling, and interpretability. It then imports all required modules.

In [ ]:
# Step 1:Install and import libraries

!pip install ydata-profiling scikit-learn xgboost imbalanced-learn shap matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML Libraries
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

# Imbalance Handling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline # I am using ImbPipleine for SMOTE compatibility

# Interpretability
import shap

# EDA
from ydata_profiling import ProfileReport

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

### Load Data and Initial Preprocessing

This cell loads the `adult_income_data.csv` file into a pandas DataFrame. It also displays basic dataset information (shape, column names, head, data types, missing values) and performs initial cleaning by stripping whitespace from column names and renaming the 'target' column to 'income'.

In [ ]:
# Step 2: Load Data
df = pd.read_csv('/content/drive/MyDrive/adult_income_data.csv')

# Dataset Info
print(f"Dataset shape: {df.shape}")
print(f"\nColumn names:\n{df.columns.tolist()}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# Strip whitespace from all column names
df.columns = df.columns.str.strip()

#Renaming 'target' column to 'income' for clarity
df.rename(columns={'target': 'income'}, inplace=True)

### Handle Missing Values and Prepare Target Variable

This cell replaces '?' characters with `np.nan` for proper missing value handling. It then separates the features (`X`) from the target variable (`y`). The 'income' target column is converted to a binary format (1 for '>50K' and 0 otherwise), and column types (numeric and categorical) are identified. Finally, it prints the dataset shape and the distribution of the target variable.

In [ ]:
df = df.replace('?', np.nan)

# Separate features and target
X = df.drop('income', axis=1)
y = df['income']

# Convert income to binary if it's text (e.g., ">50K" -> 1, "<=50K" -> 0)
if y.dtype == 'object':
    # Strip whitespace from values before conversion
    y = y.apply(lambda x: 1 if x.strip() == '>50K' else 0)

# Identify column types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Dataset Shape: {X.shape}")
print(f"Target Distribution:\n{y.value_counts(normalize=True)}")

### Exploratory Data Analysis (EDA) Report

This cell generates a comprehensive EDA report using `ydata-profiling`. The report provides insights into the data distribution, missing values, correlations, and other key statistics, helping to understand the dataset better.

In [ ]:
# Step 3:EDA
profile = ProfileReport(df, title="Adult Income EDA Report", minimal=True)
profile.to_notebook_iframe()

### Preprocessing Pipelines

This cell defines the preprocessing steps for both numerical and categorical features. Numerical features are imputed using the median and scaled using `StandardScaler`. Categorical features are imputed using the most frequent value and then one-hot encoded. These steps are combined into a `ColumnTransformer` for efficient application.

In [ ]:
# Step 4:Preprocessing Pipelines
# Numeric: Impute (Median) + Scale
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical: Impute (Most Frequent) + OneHotEncode
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

### Train-Test Split

This cell splits the dataset into training and testing sets (`X_train`, `X_test`, `y_train`, `y_test`). A `test_size` of 20% is used, and `stratify=y` ensures that the proportion of target classes is maintained in both sets, which is crucial for imbalanced datasets.

In [ ]:
# Step 5:Train-Test Split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### Define Models with SMOTE

This cell defines three machine learning models: Logistic Regression, Random Forest, and XGBoost. Each model is wrapped in an `ImbPipeline` that includes the defined `preprocessor` and `SMOTE` for handling class imbalance. SMOTE is applied only to training folds during cross-validation, preventing data leakage.

In [ ]:
# Step 6: Define Models with SMOTE

# We use ImbPipeline to ensure SMOTE is applied ONLY to training folds during CV and before scaling/encoding in the pipeline.

models = {
    'Logistic Regression': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', LogisticRegression(max_iter=1000))
    ]),

    'Random Forest': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(random_state=42))
    ]),

    'XGBoost': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
    ])
}

results = {}

### Train, Evaluate & Cross-Validate SMOTE Models

This cell iterates through the defined models, trains each on the `X_train` and `y_train` data, and evaluates their performance on the `X_test` and `y_test` sets. It calculates the ROC-AUC score, performs 5-fold cross-validation, and prints a detailed classification report for each model. The results are stored in a dictionary for later comparison.

In [ ]:
# Step 7: Train, Evaluate & Cross-Validate

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")

    # Fit the model
    model.fit(X_train, y_train)

    # Predictions
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Metrics
    roc_auc = roc_auc_score(y_test, y_pred_proba)

    # Cross-Validation (5-fold)
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')

    results[name] = {
        'model': model,
        'roc_auc': roc_auc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std(),
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }

    print(f"Test ROC-AUC: {roc_auc:.4f}")
    print(f"CV Mean ROC-AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")
    print(f"\nClassification Report:\n{classification_report(y_test, y_pred)}")

### Train and Evaluate Baseline Random Forest Classifier

This cell defines, trains, and evaluates a baseline `RandomForestClassifier` without incorporating SMOTE. This provides a point of comparison to assess the impact of SMOTE on model performance. Similar metrics (ROC-AUC, cross-validation scores, classification report) are calculated and stored.

### Baseline Model: Random Forest Classifier (Without SMOTE)

Establish a baseline using a RandomForestClassifier without applying SMOTE. This will allow me to compare the performance gains (or losses) from using oversampling techniques.

In [ ]:
# Define the baseline RandomForestClassifier pipeline (without SMOTE)
baseline_rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

print("\n==================================================")
print("Training Baseline Random Forest...")

# Fit the baseline model
baseline_rf_model.fit(X_train, y_train)

# Predictions
y_pred_proba_baseline = baseline_rf_model.predict_proba(X_test)[:, 1]
y_pred_baseline = (y_pred_proba_baseline >= 0.5).astype(int)

# Metrics
roc_auc_baseline = roc_auc_score(y_test, y_pred_proba_baseline)

# Cross-Validation (5-fold)
cv_scores_baseline = cross_val_score(baseline_rf_model, X_train, y_train, cv=5, scoring='roc_auc')

print(f"Test ROC-AUC (Baseline RF): {roc_auc_baseline:.4f}")
print(f"CV Mean ROC-AUC (Baseline RF): {cv_scores_baseline.mean():.4f} (+/- {cv_scores_baseline.std()*2:.4f})")
print(f"\nClassification Report (Baseline RF):\n{classification_report(y_test, y_pred_baseline)}")

# Store baseline results for later comparison
results['Baseline Random Forest'] = {
    'model': baseline_rf_model,
    'roc_auc': roc_auc_baseline,
    'cv_mean': cv_scores_baseline.mean(),
    'cv_std': cv_scores_baseline.std(),
    'y_pred': y_pred_baseline,
    'y_pred_proba': y_pred_proba_baseline
}

### Compare Model Performance

This cell prints a summary of the performance for all trained models, including those with SMOTE and the baseline. It displays the test ROC-AUC score and the mean cross-validation ROC-AUC score for each model, allowing for easy comparison.

### Comparison of Model Performance (with and without SMOTE)

The ROC-AUC scores for all trained models to see the impact of SMOTE and identify the best performing model.

In [ ]:
print("\n==================================================")
print("Model Performance Comparison")
print("==================================================")

for name, metrics in results.items():
    print(f"\nModel: {name}")
    print(f"  Test ROC-AUC: {metrics['roc_auc']:.4f}")
    print(f"  CV Mean ROC-AUC: {metrics['cv_mean']:.4f} (+/- {metrics['cv_std']*2:.4f})")


### Feature Importance Analysis for Best Model (XGBoost)

This cell performs a feature importance analysis on the best-performing model (XGBoost). It extracts feature names after preprocessing, retrieves feature importances from the trained classifier, and displays the top 20 most important features in a DataFrame. This helps to understand which features contribute most to the model's predictions.

### Feature Importance Analysis for the Best Model (XGBoost)

Features that the best-performing model, XGBoost, considered most important for predicting income. This analysis will provide insights into which attributes significantly influence the model's decisions.

In [ ]:
# Get the best performing model (XGBoost)
best_model = results['XGBoost']['model']

# Get feature names after preprocessing
# For numerical features, use original names
processed_feature_names = numeric_features.copy()

# For categorical features, get one-hot encoded names
onehot_features = best_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)
processed_feature_names.extend(onehot_features)

# Extract feature importances from the XGBoost classifier
feature_importances = best_model.named_steps['classifier'].feature_importances_

# Create a DataFrame for better visualization
importance_df = pd.DataFrame({
    'Feature': processed_feature_names,
    'Importance': feature_importances
})

# Sort by importance in descending order
importance_df = importance_df.sort_values(by='Importance', ascending=False)

# Display the top 20 most important features
print("\nTop 20 Most Important Features:\n")
display(importance_df.head(20))

### Visualize Top 20 Feature Importances

This cell generates a bar plot visualizing the top 20 feature importances derived from the XGBoost model. This provides a clear graphical representation of the most influential features.

In [ ]:
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=importance_df.head(20), palette='viridis')
plt.title('Top 20 Feature Importances (XGBoost)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### GridSearchCV for Hyperparameter Tuning (Optional)

This cell demonstrates an optional `GridSearchCV` to find the optimal hyperparameters for the best model (XGBoost in this case). It defines a parameter grid and performs a grid search with cross-validation to identify the combination of hyperparameters that yields the best ROC-AUC score.

In [ ]:
# STEP 8: GridSearchCV (Optional - Run on Best Model)

best_model_name = max(results, key=lambda x: results[x] ['roc_auc'])
print(f"\n{'='*50}")
print(f"Best Model: {best_model_name}")

# Example: GridSearchCV for XGBoost (Adjust for other models if needed)
if best_model_name == 'XGBoost':
    param_grid = {
        'classifier__n_estimators': [100, 200],
        'classifier__max_depth': [3, 5],
        'classifier__learning_rate': [0.01, 0.1],
        'classifier__subsample': [0.8, 1.0]
    }

    grid_search = GridSearchCV(
        models['XGBoost'],
        param_grid,
        cv=3,
        scoring='roc_auc',
        n_jobs=-1,
        verbose=1
    )
    print("Running GridSearchCV...")
    grid_search.fit(X_train, y_train)

    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV Score: {grid_search.best_score_:.4f}")

    # Update best model
    best_model = grid_search.best_estimator_
else:
    best_model = results[best_model_name] ['model']

### Generate Learning Curves

This cell generates learning curves for all defined models. Learning curves help diagnose bias-variance trade-offs by plotting the model's performance (ROC-AUC) on both training and validation sets as the number of training examples increases. This can indicate if the model would benefit from more data or different complexity.

In [ ]:
# STEP 9: Learning Curves

print("\nGenerating Learning Curves...")

plt.figure(figsize=(10, 8))
for name, model in models.items():
    # We need to use the pipeline without SMOTE for learning_curve if SMOTE is included
    # But since SMOTE is in the pipeline, we pass the full pipeline.
    # Note: SMOTE in learning_curve can be computationally expensive.

    train_sizes, train_scores, test_scores = learning_curve(
        model, X, y, cv=5, scoring='roc_auc',
        train_sizes=np.linspace(0.1, 1.0, 10),
        n_jobs=-1,
        shuffle=True,
        random_state=42
    )

    train_mean = np.mean(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)

    plt.plot(train_sizes, train_mean, 'o-', label=f'{name} (Train)')
    plt.plot(train_sizes, test_mean, 's-', label=f'{name} (Val)')

plt.xlabel('Training Examples')
plt.ylabel('ROC-AUC Score')
plt.title('Learning Curves')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### SHAP Values for Model Interpretability

This cell calculates and visualizes SHAP (SHapley Additive exPlanations) values for the best model. SHAP values help explain individual predictions by showing the contribution of each feature to the output. It generates a bar plot of mean absolute SHAP values (overall feature importance) and a bee swarm plot to show the distribution of SHAP values across the dataset for each feature.

In [ ]:
# STEP 10: SHAP Values (Interpretability)
print("\nGenerating SHAP Values (this may take a moment)...")

# Use the best model (ensure it's fitted)
# We need to transform the test set to get the feature names for SHAP
X_test_processed = best_model.named_steps['preprocessor'].transform(X_test)

# Get feature names after preprocessing
ohe = best_model.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
cat_feature_names = ohe.get_feature_names_out(categorical_features)
all_feature_names = np.concatenate([numeric_features, cat_feature_names])

# Create SHAP explainer
# For XGBoost, use TreeExplainer; for others, use KernelExplainer (slower)
if 'XGBClassifier' in str(type(best_model.named_steps['classifier'])):
    explainer = shap.TreeExplainer(best_model.named_steps['classifier'])
    shap_values = explainer.shap_values(X_test_processed)
else:
    # Fallback for non-tree models (slower)
    explainer = shap.KernelExplainer(best_model.named_steps['classifier'].predict_proba, X_test_processed[:100])
    shap_values = explainer.shap_values(X_test_processed)

# Handle case where shap_values might be a list (for binary classification)
if isinstance(shap_values, list):
    shap_values = shap_values  # Get values for the positive class (>50K)

# Plot
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test_processed, feature_names=all_feature_names, plot_type="bar")
plt.tight_layout()
plt.show()

# Bee swarm plot
plt.figure(figsize=(12, 10))
shap.summary_plot(shap_values, X_test_processed, feature_names=all_feature_names)
plt.tight_layout()
plt.show()

# Print top features
print("\nTop 10 Features by SHAP Importance:")
feature_importance = np.abs(shap_values).mean(axis=0)
top_features = pd.DataFrame({
    'feature': all_feature_names,
    'shap_importance': feature_importance
}).sort_values('shap_importance', ascending=False).head(10)
print(top_features)


In [1]:
%cd "/content/drive/MyDrive/Colab Notebooks"

/content/drive/MyDrive/Colab Notebooks
